# Week 4 Practice: Model Evaluation and Optimization

## Objective
Evaluate machine learning models using cross-validation, optimize hyperparameters using GridSearchCV, and save and load trained models using joblib.

## Tasks
1. Use cross_val_score to evaluate model accuracy.
2. Implement GridSearchCV for an SVM classifier.
3. Save the trained model using joblib.
4. Load the saved model and predict new data.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

import joblib

In [2]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print("Dataset shape:", X.shape)
print("Target names:", data.target_names)

print("\nFirst five rows:")
print(X.head())

print("\nClass distribution:")
print(pd.Series(y).value_counts())

Dataset shape: (569, 30)
Target names: ['malignant' 'benign']

First five rows:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean frac

In [3]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", random_state=42))
])

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation scores:", scores)
print("Mean accuracy:", round(scores.mean(), 4))
print("Mean accuracy (%):", round(scores.mean() * 100, 2), "%")
print("Standard deviation:", round(scores.std(), 4))

Cross-validation scores: [0.97368421 0.95614035 1.         0.96491228 0.97345133]
Mean accuracy: 0.9736
Mean accuracy (%): 97.36 %
Standard deviation: 0.0147


### Observation

Five-fold cross-validation was used to evaluate the SVM classifier on the Breast Cancer Wisconsin dataset. The accuracy of each fold and the mean accuracy were calculated. Cross-validation provides a more reliable estimate of model performance than evaluating the model using only one train-test split.

In [5]:
param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__kernel": ["linear", "rbf"],
    "svm__gamma": ["scale", "auto"]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X, y)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(round(grid_search.best_score_, 4))

print("\nBest Accuracy (%):")
print(round(grid_search.best_score_ * 100, 2), "%")


Best Parameters:
{'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}

Best Cross-Validation Accuracy:
0.9772

Best Accuracy (%):
97.72 %


In [6]:
print("Original SVM mean CV accuracy:",
      round(scores.mean(), 4))

print("Optimized SVM mean CV accuracy:",
      round(grid_search.best_score_, 4))

improvement = (
    grid_search.best_score_ - scores.mean()
) * 100

print("Accuracy difference:",
      round(improvement, 2), "percentage points")

Original SVM mean CV accuracy: 0.9736
Optimized SVM mean CV accuracy: 0.9772
Accuracy difference: 0.35 percentage points


### Observation

GridSearchCV was used to identify suitable SVM hyperparameters by evaluating different combinations of C, kernel, and gamma using five-fold cross-validation. The best parameter combination and its mean validation accuracy were recorded. The optimized model can then be used for prediction and model persistence.

In [7]:
# Get the best trained pipeline from GridSearchCV
best_model = grid_search.best_estimator_

# Save the trained model
joblib.dump(best_model, "svm_best_model.joblib")

print("Model saved successfully!")

Model saved successfully!


In [8]:
loaded_model = joblib.load("svm_best_model.joblib")

print("Model loaded successfully!")

Model loaded successfully!


In [9]:
# Select one example
new_data = X.iloc[[0]]

# Predict using the loaded model
prediction = loaded_model.predict(new_data)

predicted_species = data.target_names[prediction[0]]

print("Predicted class:", predicted_species)
print("Actual class:", data.target_names[y[0]])

Predicted class: malignant
Actual class: malignant


## Conclusion

Cross-validation was used to evaluate the SVM classifier, and GridSearchCV was applied to select suitable hyperparameters. The best trained pipeline was saved using joblib and successfully loaded for prediction on new input data. This practice demonstrates model evaluation, hyperparameter optimization, and model persistence, which are important steps in building reusable machine learning workflows.